In [29]:
import pandas as pd
import numpy as np
from cca_classes import LogNormalJumpCCAPricer, BaselineCCAPricer, compute_lcl_usd, compute_barrier_kvm

## Baseline Model Panel

In [74]:

study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']


cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T = 5.0
vol_window = 52
freq = 'W'
ann_factor = np.sqrt(52)


cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)


## M2 Specific Data

In [75]:
ovx_df = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')
ovx_df['date'] = pd.to_datetime(ovx_df['date'])
ovx_df = ovx_df.sort_values('date')
ovx_df['OVXCLS'] = ovx_df['OVXCLS'].ffill()

cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

cca_panel_df = pd.merge_asof(
    cca_panel_df,
    ovx_df[['date', 'OVXCLS']],
    on='date',
    direction='backward'
)

cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
print(f"OVX NAs: {cca_panel_df['OVXCLS'].isna().sum()}")

OVX NAs: 0


## M2 Parameters & Pricer

In [76]:
gamma      = 0.15
mu_J       = -0.016617
sigma_J    =  0.099499

pricer = LogNormalJumpCCAPricer(
    mu_J=mu_J * gamma,
    sigma_J=sigma_J * gamma,
    max_terms=150
)
pricer._cache = {}
baseline_pricer = BaselineCCAPricer()

# sigmoid parameters from jump_estimation.ipynb
L  = 0.2868   # λ_max weekly
k  = 0.1117   # steepness
x0 = 60.0     # OVX midpoint

def ovx_to_lambda(ovx):
    if np.isnan(ovx) or ovx <= 0:
        return 0.0
    lam_weekly = L / (1 + np.exp(-k * (ovx - x0)))
    return lam_weekly * 52

In [77]:
def prepare_country(df):
    df = df.copy().sort_values('date').reset_index(drop=True)

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            df['monetary_base_bn_local'],
            df['domestic_debt_bn_local'],
            df['fx_rate'],
            df['domestic_rate'],
            df['risk_free_rate']
        )
    ]

    log_ret         = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(df['external_debt_bn_usd'], df['risk_free_rate'])
    ]

    df['lambda_annual'] = df['OVXCLS'].apply(ovx_to_lambda)

    return df

## Run M2

In [78]:
results = []
print("Starting M2 calibration...")

for country, group in cca_panel_df.groupby('country'):
    print(f"Processing {country}...")
    df = prepare_country(group)

    implied_V       = []
    implied_sigma_diff = []
    implied_sigma_V = []
    cca_converged   = []

    prev = None

    for i, row in df.iterrows():
        lam = row['lambda_annual']

        cca = pricer.solve_CCA_M2(
            row['LCL_usd'], row['sigma_lcl'], row['B_f'],
            row['risk_free_rate'], T, lam,
            prev_solution=prev
        )

        prev = cca if cca['converged'] else prev

        implied_V.append(cca['implied_V'])
        implied_sigma_diff.append(cca['implied_sigma_diff'])
        implied_sigma_V.append(cca['implied_sigma_V'])
        cca_converged.append(cca['converged'])

    df['implied_V']          = implied_V
    df['implied_sigma_diff'] = implied_sigma_diff
    df['implied_sigma_V']    = implied_sigma_V
    df['cca_converged']      = cca_converged

    results.append(df)

results = pd.concat(results, ignore_index=True)

START_DATE = '2015-01-01'
END_DATE   = '2024-12-31'
results    = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

print(f"Calibration complete!")
print(f"Converged: {results['cca_converged'].sum()} / {len(results)}")
print(f"Convergence rate: {results['cca_converged'].mean()*100:.1f}%")

Starting M2 calibration...
Processing Brazil...
Processing Chile...
Processing China...
Processing Colombia...
Processing Egypt...
Processing Indonesia...
Processing Malaysia...
Processing Mexico...
Processing Philippines...
Processing Qatar...
Processing Saudi Arabia...
Processing South Africa...
Processing South Korea...
Processing Thailand...
Processing Turkey...
Processing UAE (Abu Dhabi)...
Calibration complete!
Converged: 8352 / 8352
Convergence rate: 100.0%


In [79]:
results[[
    'date', 'country', 'cds_spread', 'risk_free_rate',
    'implied_V', 'implied_sigma_V', 'cca_converged',
    'B_f', 'LCL_usd', 'sigma_lcl', 'lambda_annual'
]].to_csv('../output/results/results_w_domesticdebt/M2_results_5YCDS_weekly_15_dampening.csv', index=False)